# 🧹 Remoção de Artefatos Preservando Trincas — Pré-processamento Seletivo

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Pré-processamento Avançado

---

## Contexto

Imagens de inspeção de obras são capturadas em condições adversas: granulação de câmera, reflexos, manchas de umidade, sujeira e variação de iluminação criam **artefatos visuais** que dificultam tanto a inspeção humana quanto algoritmos de detecção. A solução óbvia — suavizar a imagem — resolve o ruído mas **apaga exatamente o que nos interessa**: a trinca, que é uma linha escura fina.

Este módulo resolve esse dilema com uma estratégia em três etapas:

```
1. Detectar a trinca → máscara binária
2. Suavizar fortemente apenas o FUNDO (fora da máscara)
3. Recombinar fundo limpo + trinca preservada
```

O resultado é uma imagem onde o ruído foi removido **sem sacrificar a geometria da patologia**.

---

## Por que Black-Hat para detectar trincas?

A morfologia matemática oferece operações que realçam estruturas específicas por forma e tamanho. O **Black-Hat** é definido como:

$$\text{BlackHat}(f) = (f \bullet B) - f$$

onde $\bullet$ é o fechamento morfológico e $B$ é o elemento estruturante.

**Intuição:** o fechamento com um kernel grande "preenche" as linhas escuras finas (a trinca), como se passasse uma esponja sobre ela. A subtração do original realça exatamente o que foi preenchido — ou seja, **as estruturas escuras menores que o kernel**.

| Operação | Realça |
|----------|--------|
| Top-Hat | Estruturas **claras** menores que o kernel |
| **Black-Hat** | Estruturas **escuras** menores que o kernel ← trincas |

### Parâmetro crítico: `bh_ksize`

O kernel deve ser **maior que a largura da trinca** e **menor que outros elementos escuros** (sombras, juntas). Valores típicos para concreto:

| Tipo de imagem | `bh_ksize` recomendado |
|----------------|------------------------|
| Close-up (trinca fina, <5 px) | 11–21 |
| Foto geral de fachada | 21–41 |
| Imagem de drone | 31–61 |

---

## Por que NLMeans + Bilateral para o fundo?

Suavizações simples (Gaussian, Median) removem ruído mas borram bordas. O pipeline usa dois filtros complementares:

| Filtro | Mecanismo | Vantagem |
|--------|-----------|----------|
| **NLMeans** (`fastNlMeansDenoisingColored`) | Compara patches similares na imagem inteira | Remove granulação preservando texturas repetitivas (concreto, agregados) |
| **Bilateral** | Pondera pixels por distância espacial *e* similaridade de cor | Preserva bordas e transições de material |

Aplicar Bilateral *após* NLMeans amplifica os pontos fortes de cada um sem acumular seus artefatos.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Testar o ambiente com imagem sintética (sem Drive) |
| 2 | Montar o Google Drive e configurar parâmetros |
| 3 | Processar todas as imagens em lote |
| 4 | Visualizar comparativos e avaliar qualidade |
| 5 | Calibrar parâmetros com grade visual |

> 💡 **Recomendação:** execute a Seção 1 primeiro — ela não depende de nenhum arquivo externo.

---
## Seção 1 — Teste com imagem sintética

A imagem abaixo simula os problemas típicos de uma foto de inspeção:

- **Fundo cinza com textura ruidosa** → concreto com granulação de câmera
- **Manchas irregulares** → umidade, sujeira
- **Trinca diagonal fina** → o que queremos **preservar**

**Resultado esperado após o processamento:**
- Fundo uniforme, ruído reduzido
- Trinca intacta na máscara e na imagem final
- Comparativo lado a lado mostrando a melhora

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── Criar imagem sintética 400×600 (BGR)
h, w = 400, 600
img_sint = np.full((h, w, 3), 155, dtype=np.uint8)

# Ruído gaussiano (granulação de câmera)
ruido = np.random.normal(0, 18, img_sint.shape).astype(np.int16)
img_sint = np.clip(img_sint.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# Manchas de umidade (blobs cinza-escuro)
for cx, cy, r in [(150, 100, 40), (420, 280, 55), (80, 300, 30)]:
    cv2.ellipse(img_sint, (cx, cy), (r, int(r*0.6)), 45, 0, 360, (110, 110, 110), -1)

# Trinca principal (diagonal fina — o que deve ser PRESERVADO)
pts = np.array([[80, 30], [200, 150], [320, 250], [480, 370]], dtype=np.int32)
for i in range(len(pts) - 1):
    cv2.line(img_sint, tuple(pts[i]), tuple(pts[i+1]), (30, 30, 30), 2)

# Ramificação
cv2.line(img_sint, (250, 200), (290, 240), (40, 40, 40), 1)

# ── Funções do pipeline
def detect_crack_mask(img_bgr, ksize=21, dilate_iters=1):
    """Detecta trinca via Black-Hat e retorna máscara binária."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)
    ksize = max(3, ksize | 1)  # garante ímpar >= 3
    se = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, se)
    _, mask = cv2.threshold(blackhat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if dilate_iters > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask = cv2.dilate(mask, k, iterations=dilate_iters)
    return mask

def smooth_background_only(img_bgr, crack_mask,
                            nlm_h=10, nlm_template=7, nlm_search=21,
                            bil_d=11, bil_sc=75, bil_ss=75):
    """Suaviza fundo preservando a região da trinca."""
    den = cv2.fastNlMeansDenoisingColored(img_bgr, None, nlm_h, nlm_h, nlm_template, nlm_search)
    sm  = cv2.bilateralFilter(den, d=bil_d, sigmaColor=bil_sc, sigmaSpace=bil_ss)
    bg_mask = cv2.bitwise_not(crack_mask)
    bg_smooth = cv2.bitwise_and(sm,      sm,      mask=bg_mask)
    fg_crack  = cv2.bitwise_and(img_bgr, img_bgr, mask=crack_mask)
    return cv2.add(bg_smooth, fg_crack)

# ── Executar
crack_mask = detect_crack_mask(img_sint, ksize=21, dilate_iters=1)
resultado  = smooth_background_only(img_sint, crack_mask)

# ── Visualizar
fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
dados = [
    (img_sint,   'Original (com artefatos)', None),
    (crack_mask, 'Máscara Black-Hat\n(trinca detectada)', 'gray'),
    (resultado,  'Fundo limpo +\ntrinca preservada', None),
    (np.hstack([img_sint, resultado]), 'Comparativo\nAntes × Depois', None),
]

for ax, (im, t, cmap) in zip(eixos, dados):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
    ax.set_title(t, fontsize=9)
    ax.axis('off')

plt.suptitle('✅ Teste de ambiente — remoção de artefatos com preservação de trinca',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Ambiente OK — pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar parâmetros

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito → Organizar → Adicionar atalho ao Drive*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
from pathlib import Path

# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/fotos_obra')
PASTA_SAIDA   = Path('/content/drive/MyDrive/fotos_obra_clean')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Parâmetros Black-Hat (detecção da trinca)
BH_KSIZE    = 21   # kernel ímpar — deve ser maior que a largura da trinca
DILATE_ITER = 1    # dilata a máscara para cobrir bordas da trinca

# ── Parâmetros de suavização do fundo
NLM_H        = 10   # força do NLMeans (maior = mais suavização, mais lento)
NLM_TEMPLATE = 7    # tamanho do patch de comparação (deve ser ímpar)
NLM_SEARCH   = 21   # janela de busca (deve ser ímpar)
BIL_D        = 11   # diâmetro do bilateral
BIL_SC       = 75   # sigmaColor — tolerância de cor
BIL_SS       = 75   # sigmaSpace — tolerância espacial

# ── Redimensionamento (0 = sem resize)
LARGURA_MAX  = 1200

# ── Extensões aceitas
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ── Verificar pasta
if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    print(f'📂 Entrada   : {PASTA_ENTRADA}')
    print(f'📁 Saída     : {PASTA_SAIDA}')
    print(f'⚙️  Black-Hat : ksize={BH_KSIZE}, dilate={DILATE_ITER}')
    print(f'⚙️  NLMeans   : h={NLM_H}, template={NLM_TEMPLATE}, search={NLM_SEARCH}')
    print(f'⚙️  Bilateral : d={BIL_D}, sigmaColor={BIL_SC}, sigmaSpace={BIL_SS}')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Processamento em lote

### Pipeline completo

```
Imagem colorida (BGR)
        │
        ├─── [DETECÇÃO DA TRINCA] ──────────────────────────────────────
        │         │
        │    cvtColor → GRAY
        │         │
        │    medianBlur (3×3)       ← remove sal-e-pimenta antes do blackhat
        │         │
        │    MORPH_BLACKHAT (ksize) ← realça linhas escuras finas
        │         │
        │    threshold (Otsu)       ← binariza automaticamente
        │         │
        │    dilate (1–2 iter)      ← expande máscara para cobrir borda da trinca
        │         │
        │       MÁSCARA
        │
        ├─── [SUAVIZAÇÃO DO FUNDO] ─────────────────────────────────────
        │         │
        │    NLMeans (coloured)     ← remove granulação preservando textura
        │         │
        │    bilateralFilter        ← remove ruído residual preservando bordas
        │         │
        │    bitwise_and(~MÁSCARA)  ← mantém apenas o fundo suavizado
        │
        ├─── [PRESERVAÇÃO DA TRINCA] ───────────────────────────────────
        │    bitwise_and(MÁSCARA)   ← extrai trinca original sem tocar
        │
        └─── cv2.add(fundo, trinca) ← recombina
                  │
             IMAGEM FINAL
```

Cada imagem gera 3 arquivos na pasta de saída:
- `*_crack_mask.png` → máscara binária da trinca
- `*_clean.png` → imagem com artefatos removidos e trinca preservada
- `*_comparativo.png` → original e resultado lado a lado

> ⏱️ **Atenção:** NLMeans é o passo mais lento. Imagens de 1200 px levam ~2–5 s cada. Para datasets grandes, considere reduzir `NLM_SEARCH` de 21 para 11.

In [ ]:
from tqdm.notebook import tqdm

erros = []

for arq in tqdm(arquivos, desc='Processando imagens'):
    try:
        img = cv2.imread(str(arq))
        assert img is not None, 'Falha ao ler'

        # Redimensionar se necessário
        if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
            r   = LARGURA_MAX / img.shape[1]
            img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

        mask      = detect_crack_mask(img, ksize=BH_KSIZE, dilate_iters=DILATE_ITER)
        resultado = smooth_background_only(
            img, mask,
            nlm_h=NLM_H, nlm_template=NLM_TEMPLATE, nlm_search=NLM_SEARCH,
            bil_d=BIL_D, bil_sc=BIL_SC, bil_ss=BIL_SS
        )

        base = arq.stem
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_crack_mask.png'), mask)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_clean.png'), resultado)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_comparativo.png'),
                    cv2.hconcat([img, resultado]))

    except Exception as e:
        erros.append((arq.name, str(e)))

print(f'\n✅ {len(arquivos) - len(erros)} imagem(ns) processada(s).')
print(f'   3 arquivos gerados por imagem em: {PASTA_SAIDA}')
if erros:
    print('\n⚠️  Erros:')
    for nome, msg in erros:
        print(f'   {nome}: {msg}')

---
## Seção 4 — Visualização e avaliação de qualidade

### 4a. Painel de quatro vistas

Para cada imagem, exibimos:

| Vista | O que observar |
|-------|----------------|
| Original | Referência — artefatos visíveis |
| Máscara Black-Hat | A trinca deve estar **branca** e o fundo **preto**. Se ruídos grandes aparecerem brancos, aumente `BH_KSIZE`. Se a trinca sumir, diminua. |
| Resultado | Fundo limpo. Trinca deve ter a mesma geometria da original. |
| Comparativo | Avaliação visual lado a lado — a textura do concreto deve estar mais uniforme sem apagar a trinca. |

In [ ]:
MAX_EXIBIR = 4

for arq in arquivos[:MAX_EXIBIR]:
    img = cv2.imread(str(arq))
    if img is None:
        continue
    if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
        r   = LARGURA_MAX / img.shape[1]
        img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0] * r)),
                         interpolation=cv2.INTER_AREA)

    mask = detect_crack_mask(img, ksize=BH_KSIZE, dilate_iters=DILATE_ITER)
    res  = smooth_background_only(
        img, mask,
        nlm_h=NLM_H, nlm_template=NLM_TEMPLATE, nlm_search=NLM_SEARCH,
        bil_d=BIL_D, bil_sc=BIL_SC, bil_ss=BIL_SS
    )

    fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
    dados = [
        (img,  'Original',           None),
        (mask, 'Máscara Black-Hat',  'gray'),
        (res,  'Resultado (limpo)',   None),
        (cv2.hconcat([img, res]), 'Comparativo', None),
    ]
    for ax, (im, t, cmap) in zip(eixos, dados):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.axis('off')

    plt.suptitle(arq.name, fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4b. Análise de diferença pixel a pixel

O mapa de diferença absoluta mostra exatamente onde a imagem mudou. Regiões brancas = muita modificação (artefatos removidos). Regiões pretas = sem alteração (trinca preservada).

**O ideal:** a trinca deve aparecer **escura** no mapa de diferença — ou seja, não foi tocada.

In [ ]:
if arquivos:
    img = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
        r   = LARGURA_MAX / img.shape[1]
        img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0] * r)),
                         interpolation=cv2.INTER_AREA)

    mask = detect_crack_mask(img, ksize=BH_KSIZE, dilate_iters=DILATE_ITER)
    res  = smooth_background_only(
        img, mask,
        nlm_h=NLM_H, nlm_template=NLM_TEMPLATE, nlm_search=NLM_SEARCH,
        bil_d=BIL_D, bil_sc=BIL_SC, bil_ss=BIL_SS
    )

    diff = cv2.absdiff(img, res)
    diff_gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    diff_norm = cv2.normalize(diff_gray, None, 0, 255, cv2.NORM_MINMAX)

    # Sobreposição da máscara em vermelho para referência
    diff_rgb = cv2.cvtColor(diff_norm, cv2.COLOR_GRAY2BGR)
    diff_rgb[mask > 0] = [0, 0, 180]  # trinca em vermelho

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.imshow(diff_norm, cmap='hot')
    ax1.set_title('Mapa de diferença (|original − resultado|)\nBranco = muito modificado', fontsize=10)
    ax1.axis('off')

    ax2.imshow(cv2.cvtColor(diff_rgb, cv2.COLOR_BGR2RGB))
    ax2.set_title('Diferença + máscara da trinca (vermelho)\nTrincas em vermelho não devem ter mudado', fontsize=10)
    ax2.axis('off')

    plt.suptitle(f'Análise de diferença — {arquivos[0].name}', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

    mod_fundo  = diff_gray[mask == 0].mean()
    mod_trinca = diff_gray[mask > 0].mean()
    print(f'Modificação média no FUNDO  : {mod_fundo:.1f} / 255')
    print(f'Modificação média na TRINCA : {mod_trinca:.1f} / 255')
    print(f'Razão fundo/trinca          : {mod_fundo/max(mod_trinca,0.1):.1f}× (quanto maior, melhor a preservação)')

---
## Seção 5 — Calibração de parâmetros

### O que ajustar primeiro?

A calibração deve seguir esta ordem:

1. **`BH_KSIZE`** — acertar a máscara da trinca é a etapa mais crítica. Se a máscara estiver errada, todo o resto fica comprometido.
2. **`DILATE_ITER`** — ajusta se a trinca está "vazando" para o fundo suavizado.
3. **`NLM_H`** — controla a força da suavização. Aumente para imagens muito ruidosas.
4. **`BIL_SC` / `BIL_SS`** — refinamento final de bordas.

A grade abaixo varia `BH_KSIZE` (linhas) × `NLM_H` (colunas) para a primeira imagem.

In [ ]:
if arquivos:
    img_ref = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_ref.shape[1] > LARGURA_MAX:
        r       = LARGURA_MAX / img_ref.shape[1]
        img_ref = cv2.resize(img_ref, (LARGURA_MAX, int(img_ref.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

    bh_ksizes = [11, 21, 31, 41]
    nlm_hs    = [5, 10, 15]

    fig, eixos = plt.subplots(len(bh_ksizes), len(nlm_hs),
                               figsize=(6 * len(nlm_hs), 5 * len(bh_ksizes)))

    for i, bk in enumerate(bh_ksizes):
        for j, nh in enumerate(nlm_hs):
            mask_g = detect_crack_mask(img_ref, ksize=bk, dilate_iters=DILATE_ITER)
            res_g  = smooth_background_only(
                img_ref, mask_g,
                nlm_h=nh, nlm_template=NLM_TEMPLATE, nlm_search=NLM_SEARCH,
                bil_d=BIL_D, bil_sc=BIL_SC, bil_ss=BIL_SS
            )
            eixos[i][j].imshow(cv2.cvtColor(res_g, cv2.COLOR_BGR2RGB))
            eixos[i][j].set_title(f'bh_ksize={bk}  nlm_h={nh}', fontsize=8)
            eixos[i][j].axis('off')

    plt.suptitle(f'Grade de parâmetros — {arquivos[0].name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('👆 Escolha a combinação com fundo mais limpo e trinca mais nítida.')
    print('   Atualize BH_KSIZE e NLM_H na Seção 2 e reprocesse.')

---
## Próximos passos

Com as imagens pré-processadas em `fotos_obra_clean`, o pipeline de visão computacional fica:

```
fotos_obra_clean/
        │
        ├── CLAHE (módulo 03)          ← realce de contraste adicional
        ├── Detecção de bordas (Canny) ← segmentação da trinca
        ├── U-Net / SAM                ← segmentação semântica
        └── Confronto com IFC          → registro BCF de não-conformidade
```

### Quando usar este pré-processamento?

| Situação | Recomendação |
|----------|--------------|
| Câmera de drone (alta ISO, muito ruído) | ✅ Aplicar antes de qualquer detecção |
| Foto de smartphone em boa luz | ⚠️ Opcional — avalie se o NLMeans não remove detalhes finos |
| Dataset para treinar U-Net | ✅ Aplicar nas imagens, mas **não** nas máscaras de anotação |
| Inspeção em tempo real (câmera IP) | ❌ NLMeans é lento demais — use bilateral isolado |

### Referências

- Serra, J. (1983). *Image Analysis and Mathematical Morphology*. Academic Press. — fundamentos do Black-Hat
- Buades, A.; Coll, B.; Morel, J-M. (2005). *A Non-Local Means Denoising Algorithm*. CVPR.
- OpenCV docs: [`morphologyEx`](https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html), [`fastNlMeansDenoisingColored`](https://docs.opencv.org/4.x/d1/d79/group__photo__denoise.html)
- ABNT NBR 6118:2014 — Projeto de estruturas de concreto: procedimento
